In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import warnings

# 데이터 불러오기

In [ ]:
train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv')
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv',header=None)
train_label = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv')
warnings.filterwarnings(action='ignore')

In [ ]:
df = train.join(train_label,how = 'left')
df.head()

# 데이터 프레임 컬럼병 변경

In [ ]:
cols_range = list(range(len(df.columns)))

df.columns = [str(i) for i in cols_range]
df.columns

In [ ]:
df['40'] = df['40'].astype(str)

In [ ]:
print(df.shape)
df.info()

# EDA

In [ ]:
feature_cols = df.columns[:-1] 
target_cols = df.columns[-1]

plt.figure(figsize = [30,15])
plt.subplots_adjust(wspace = 1,hspace = 1)
for i,var in enumerate(feature_cols):
    plt.subplot(5,8,i+1)
    sns.kdeplot(data = df,
                x = var,
                hue = target_cols,
                palette = ['red','dodgerblue'],
                label = target_cols)
    plt.title(f"kde_plot of {var}")
    plt.legend(loc='upper right',  prop={'size': 6})

plt.show()

### 전부 평균이 0인 정규분포를 따르는 듯 하고, target 값과 target값이 아닌 데이터들의 확률 밀도 그래프가 큰 차이가 보이지 않음

In [ ]:
df['40'].hist()
plt.title('hisogram of target')

# target 값들도 각 500개 씩 비율이 비슷하다

# 상관관계 보기

In [ ]:
mask = np.triu(np.ones_like(df.corr()))

plt.figure(figsize = [20,20])
sns.heatmap(df.corr(), mask = mask, cmap = 'binary', vmin = -1, vmax = 1, annot = True, fmt = '.1')

# 독립변수들간 상관관계가 뚜렷하게 보이지 않는다.
### 몇 가지만 -0.5~ 0.5 사이로 상관관계가 보이는 것이 있어 상관관계가 보이는 독립변수들만 보도록 함 

In [ ]:
corr = df.corr()

cond = corr[(corr > 0.5) | (corr < -0.5)]

plt.figure(figsize = [20,20])
sns.heatmap(cond, mask = mask, cmap = 'binary', vmin = -1, vmax = 1, annot = True, fmt = '.1')

In [ ]:
corr_cols = ['4','12','23','28','40']

sns.pairplot(df.loc[:,corr_cols], hue = '40',corner = True)

# 12번과 28번, 4번과 28번, 4번과 23번 정도가 선형성을 보이는 것을 확인 가능

# Modeling 

In [ ]:
x_train,y_train,x_test,y_test = train_test_split(df.iloc[:,:-1], df.iloc[:,-1], test_size = 0.2)

# 10개의 validation set 으로 나눠 평가 후 최적의 K 개를 찾도록 함 

In [ ]:
cv = KFold(n_splits = 10)

valid_df = x_train.join(x_test)

avg_score = []

for k in range(1,20):
    model = KNeighborsClassifier(n_neighbors = k)
    score_list = []

    for v,t in cv.split(valid_df):
        v_train_feature = valid_df.iloc[v,:-1]
        v_train_target = valid_df.iloc[v,-1]
        v_test_feature = valid_df.iloc[t,:-1]
        v_test_target = valid_df.iloc[t,-1]
        
        model.fit(v_train_feature, v_train_target)
        score = model.score(v_test_feature,v_test_target)

        
        score_list.append(score)
        
    avg_score.append(sum(score_list) / len(score_list))
    
    
avg_score

In [ ]:
plt.figure(figsize = [10,5])
plt.plot(avg_score, ls = '--', c = 'r')
plt.axvline (x = avg_score.index(max(avg_score)), c = 'b', label = avg_score.index(max(avg_score)))
plt.axhline (y = max(avg_score), label = 'accuracy', c ='b')
plt.ylim = [0.65 , 1]
plt.legend()
plt.title(f'best_k is {avg_score.index(max(avg_score))} \n accuracy = {round(max(avg_score),3)}')

## 본래 K는 홀수 개로 하는 것이 맞으나, 현재는 4개의 이웃이 가장 적합하다고 판단되어 K는 4개로 모델 설정 후 예측

In [ ]:
model = KNeighborsClassifier(n_neighbors = 4)

In [ ]:
model.fit(df.iloc[:,:-1], df.iloc[:,-1])

In [ ]:
predict = model.predict(test)

idx = list(range(1,len(predict)+1))


submission = pd.DataFrame({'Solution': predict,
                           'Id':idx})

submission

In [ ]:
submission.to_csv('submission.csv',index = False)
submission = pd.read_csv('/kaggle/working/submission.csv')
submission